In [1]:
import os
from dotenv import load_dotenv
from copy import deepcopy
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))

In [5]:
import vllm
import torch
from transformers import AutoTokenizer

# --- 1. Load the vLLM engine and the model's tokenizer ---
# The tokenizer contains the chat template.
MODEL_NAME = "Qwen/Qwen3-0.6B"

llm = vllm.LLM(
    model=MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.bfloat16,
	gpu_memory_utilization=0.04
)

# It's crucial to load the tokenizer for the specific model you are using.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# --- 2. Define your conversation in a structured list ---
# This is a clean, model-agnostic way to represent a conversation.
messages = [
    {"role": "system", "content": "You are a helpful assistant based in Bandung, Indonesia."},
    {"role": "user", "content": "What are some good places to visit nearby for a weekend trip?"}
]

# --- 3. Apply the chat template to format the prompt ---
# This is the key step. apply_chat_template does all the heavy lifting.
# - tokenize=False: Returns a formatted string, which is what vLLM needs.
# - add_generation_prompt=True: Adds the tokens to signal the start of the assistant's turn (e.g., "<|im_start|>assistant\n").
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Let's see what the formatted prompt looks like:
print("--- Formatted Prompt for vLLM ---")
print(prompt)
print("-" * 35)

# --- 4. Use the formatted prompt with vLLM ---
sampling_params = vllm.SamplingParams(
    temperature=0.7,
    top_p=0.95,
    max_tokens=500,
    stop=["<|im_end|>"]
)

outputs = llm.generate([prompt], sampling_params)

# --- 5. Print the result ---
for output in outputs:
    generated_text = output.outputs[0].text
    print("--- Model Response ---")
    print(generated_text.strip())

INFO 02-20 07:44:26 [utils.py:261] non-default args: {'trust_remote_code': True, 'dtype': torch.bfloat16, 'gpu_memory_utilization': 0.04, 'disable_log_stats': True}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-20 07:44:26 [model.py:541] Resolved architecture: Qwen3ForCausalLM
INFO 02-20 07:44:26 [model.py:1561] Using max model len 40960
INFO 02-20 07:44:26 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=35414) INFO 02-20 07:44:27 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='Qwen/Qwen3-0.6B', speculative_config=None, tokenizer='Qwen/Qwen3-0.6B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, rea

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.88it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.87it/s]
(EngineCore_DP0 pid=35414) 


(EngineCore_DP0 pid=35414) INFO 02-20 07:45:00 [default_loader.py:291] Loading weights took 0.56 seconds
(EngineCore_DP0 pid=35414) INFO 02-20 07:45:01 [gpu_model_runner.py:4130] Model loading took 1.12 GiB memory and 30.957382 seconds
(EngineCore_DP0 pid=35414) INFO 02-20 07:45:10 [backends.py:812] Using cache directory: /home/hanif_zhafran07/.cache/vllm/torch_compile_cache/f227e3e026/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=35414) INFO 02-20 07:45:10 [backends.py:872] Dynamo bytecode transform time: 8.56 s
(EngineCore_DP0 pid=35414) INFO 02-20 07:45:23 [backends.py:302] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=35414) INFO 02-20 07:45:32 [backends.py:319] Compiling a graph for compile range (1, 8192) takes 17.68 s
(EngineCore_DP0 pid=35414) INFO 02-20 07:45:32 [monitor.py:34] torch.compile takes 26.24 s in total
(EngineCore_DP0 pid=35414) ERROR 02-20 07:45:32 [core.py:946] EngineCore failed to start.
(EngineCore_DP0 pid=35414) 

(EngineCore_DP0 pid=35414) Process EngineCore_DP0:
(EngineCore_DP0 pid=35414) Traceback (most recent call last):
(EngineCore_DP0 pid=35414)   File "/home/hanif_zhafran07/absa-agent/.venv/lib/python3.12/site-packages/vllm/v1/worker/gpu_model_runner.py", line 4812, in _dummy_sampler_run
(EngineCore_DP0 pid=35414)     sampler_output = self.sampler(
(EngineCore_DP0 pid=35414)                      ^^^^^^^^^^^^^
(EngineCore_DP0 pid=35414)   File "/home/hanif_zhafran07/absa-agent/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
(EngineCore_DP0 pid=35414)     return self._call_impl(*args, **kwargs)
(EngineCore_DP0 pid=35414)            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore_DP0 pid=35414)   File "/home/hanif_zhafran07/absa-agent/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1786, in _call_impl
(EngineCore_DP0 pid=35414)     return forward_call(*args, **kwargs)
(EngineCore_DP0 pid=35414)            ^^^^^^^^^^^^^^^^^^^^

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}